# GMP cache warming — N=2

This notebook incrementally builds the GMP coefficient cache for $N=2$.

## Why this notebook exists

Computing `GMPC(lam, mu)` for a new multi-partition `lam` requires finding the
kernel of the eigenvalue-equation matrix at total degree `|lam|`, which is
expensive. The result is cached in the global `GMPC_cache` dict so that
subsequent calls at the same degree are instant (cache hits).

This notebook populates the cache up to a chosen target degree `d_target`
and saves it to disk. On the next session, load the saved cache and increment
`d_target` to extend coverage.

## How the cache is filled

For a given multi-partition `lam`, calling `GMPC(lam, mu)` for *any* `mu`
computes the full kernel at degree `|lam|` and stores **all** coefficients
`GMPC(lam, *)` as a side effect.  We therefore call `GMPC(mu, mu)` (the
diagonal entry) purely as a cheap entry point to trigger this side effect.

We filter to *reduced* multi-partitions (those with non-empty last component)
via `is_reduced`, because `GMPC` handles non-reduced multi-partitions by
recursing to a shorter tuple — they add no new kernel computations and
would be redundant here.

In [ ]:
# from gmplib import *
load("../gmplib.py")

N = 2
gmp_init(load_cache("GMP" + str(N)))

## Cache warming

Set `d_target` to the next degree to compute, then run this cell.
Increment `d_target` and re-run each session to extend coverage.

Each line of output shows the multi-partition being processed and the
wall-clock time for that kernel computation.  Times are typically
sub-millisecond for degrees up to ~8 (cache already warm), rising to
seconds or more at higher degrees.

In [ ]:
d_target = 8

t_start = time.perf_counter()
n_computed = 0

for mu in filter(is_reduced, mPartitions(N, d_target)):
    t0 = time.perf_counter()
    GMPC(mu, mu)  # entry point: fills all GMPC(mu, *) as a side effect
    print(f"{mu}  {time.perf_counter() - t0:.4f}s")
    n_computed += 1

t_total = time.perf_counter() - t_start
print(f"\ndone: {n_computed} reduced multi-partitions at d={d_target} in {t_total:.2f}s")

## Save cache

Always run this cell after the warming loop completes successfully.
The cache is saved to `GMP2.sobj` in the current working directory.

In [ ]:
save_cache("GMP" + str(N))